# DR Candidate Scraper — REFINED v02
## Adapted for Folketing (Parliamentary) Election

Scrapes DR's **Folketing** ballot pages:
`https://www.dr.dk/nyheder/politik/folketingsvalg/din-stemmeseddel/kandidater/{id}-{slug}`

Key changes from v01 (kommunalvalg):
- ✅ Updated URL structure — `/folketingsvalg/` path, no `/kommune/` infix
- ✅ Cookie-consent / login-wall detection & handling via Selenium
- ✅ `vis alle` click to expand all test answers
- ✅ Improved Om-section & mærkesager extraction for the Folketing page layout
- ✅ `stemmekreds` (constituency) captured instead of municipality
- ✅ Example candidate: **Marcus Vesterager** (id 882)

### Installation

In [1]:
%pip install selenium webdriver-manager beautifulsoup4 pandas lxml


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    ElementClickInterceptedException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json
from urllib.parse import urljoin
from typing import Dict, List, Any, Optional

print('✅ Libraries imported')

✅ Libraries imported


## 2. Define Target URL and Request Headers

URL structure differences between election types:

| | kommunalvalg (v01) | **folketingsvalg (v02)** |
|---|---|---|
| Candidate URL | `/kommunalvalg/.../kandidater/kommune/{id}-{slug}` | `/folketingsvalg/.../kandidater/{id}-{slug}` |
| Listing page  | `/kommunalvalg/.../din-stemmeseddel/{muni_id}` | `/folketingsvalg/.../stemmekreds/{kreds_id}` |

In [3]:
BASE_URL = 'https://www.dr.dk'

# ── Folketing URL templates ───────────────────────────────────────────────────
CANDIDATE_BASE   = f'{BASE_URL}/nyheder/politik/folketingsvalg/din-stemmeseddel/kandidater'
STEMMEKREDS_BASE = f'{BASE_URL}/nyheder/politik/folketingsvalg/din-stemmeseddel/stemmekreds'

# Example candidate – Marcus Vesterager, id 882
EXAMPLE_CANDIDATE_URL = f'{CANDIDATE_BASE}/882-marcus-vesterager'

# Request headers (used as reference; Selenium sets its own headers automatically)
REQUEST_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'da-DK,da;q=0.9,en;q=0.8',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

print('Target URL:', EXAMPLE_CANDIDATE_URL)

Target URL: https://www.dr.dk/nyheder/politik/folketingsvalg/din-stemmeseddel/kandidater/882-marcus-vesterager


## 3. Initialize WebDriver

In [4]:
def setup_driver(headless: bool = True) -> webdriver.Chrome:
    """
    Set up Chrome WebDriver with anti-detection options.
    Set headless=False if the page opens a login modal requiring manual sign-in.
    """
    options = Options()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--window-size=1920,1080')
    options.add_argument(
        '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Mask navigator.webdriver fingerprint
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver


driver = setup_driver(headless=True)
print('✅ WebDriver initialised')

✅ WebDriver initialised


## 4. Fetch and Parse HTML Content

Includes cookie-consent dismissal and login-wall detection.
DR's Folketing stemmeseddel may redirect to `login.dr.dk` in headless mode.  
If that happens, set `headless=False` in `setup_driver()`, log in once, then re-run.

In [5]:
def dismiss_cookie_consent(driver, timeout: int = 6) -> bool:
    """Accept DR's cookie-consent banner if present."""
    xpaths = [
        "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ','abcdefghijklmnopqrstuvwxyzæøå'), 'accepter alle')]",
        "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ','abcdefghijklmnopqrstuvwxyzæøå'), 'tillad alle')]",
        "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ','abcdefghijklmnopqrstuvwxyzæøå'), 'godkend')]",
        "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ','abcdefghijklmnopqrstuvwxyzæøå'), 'accept all')]",
    ]
    for xpath in xpaths:
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            driver.execute_script('arguments[0].click();', btn)
            print('  ✓ Cookie consent dismissed')
            time.sleep(1.5)
            return True
        except (TimeoutException, Exception):
            continue
    return False


def check_login_wall(driver) -> bool:
    """Return True when DR's SSO login wall is detected."""
    if 'login.dr.dk' in driver.current_url or 'oidc/authorize' in driver.current_url:
        print('  ⚠  Login wall detected — page requires a DR account.')
        print('     → Re-run setup_driver(headless=False), log in, then resume.')
        return True
    try:
        driver.find_element(
            By.XPATH,
            "//a[contains(@href,'login.dr.dk')] | "
            "//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ',"
            "'abcdefghijklmnopqrstuvwxyz'),'log ind')]"
        )
        print('  ⚠  In-page login prompt detected.')
        return True
    except NoSuchElementException:
        pass
    return False


def load_page(driver, url: str, wait_time: int = 15) -> BeautifulSoup:
    """
    Navigate to *url*, handle cookie consent, detect login wall,
    scroll to trigger lazy-loads, and return a parsed BeautifulSoup.
    """
    driver.get(url)
    WebDriverWait(driver, wait_time).until(
        EC.presence_of_element_located((By.TAG_NAME, 'body'))
    )
    time.sleep(2)

    dismiss_cookie_consent(driver)

    if check_login_wall(driver):
        raise RuntimeError('Login wall detected — cannot scrape without authentication.')

    # Scroll to trigger lazy-loaded content
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(1)
    driver.execute_script('window.scrollTo(0, 0);')
    time.sleep(1)

    return BeautifulSoup(driver.page_source, 'lxml')


print('✅ Page-loading helpers defined')

✅ Page-loading helpers defined


## 5. Extract Candidate Personal Information

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# PRIMARY STRATEGY: extract __NEXT_DATA__ JSON embedded by Next.js
# DR's Folketing stemmeseddel is a Next.js app that serialises all page props
# into <script id="__NEXT_DATA__" type="application/json"> — this is the most
# reliable source for candidate fields.
# ─────────────────────────────────────────────────────────────────────────────

def extract_next_data(soup: BeautifulSoup) -> dict:
    """Parse __NEXT_DATA__ JSON block. Returns {} if not found."""
    tag = soup.find('script', id='__NEXT_DATA__')
    if tag and tag.string:
        try:
            return json.loads(tag.string)
        except json.JSONDecodeError:
            pass
    for tag in soup.find_all('script', type='application/json'):
        try:
            d = json.loads(tag.string or '')
            if 'props' in d or 'pageProps' in d:
                return d
        except Exception:
            pass
    return {}


def _search_dict(obj, target_key, max_depth=10):
    """Recursively find all values for target_key in a nested dict/list."""
    results = []
    if max_depth == 0:
        return results
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == target_key:
                results.append(v)
            else:
                results.extend(_search_dict(v, target_key, max_depth - 1))
    elif isinstance(obj, list):
        for item in obj:
            results.extend(_search_dict(item, target_key, max_depth - 1))
    return results


# ─────────────────────────────────────────────────────────────────────────────
# Basic info
# ─────────────────────────────────────────────────────────────────────────────

def extract_basic_info(soup: BeautifulSoup, url: str, next_data: dict = None) -> Dict[str, str]:
    """
    Extract candidate_id, name, party, stemmekreds.
    Primary: __NEXT_DATA__ JSON.  Fallback: title/meta/heading tags.
    """
    slug = url.rstrip('/').split('/')[-1]
    candidate_id = slug.split('-')[0] if '-' in slug else slug
    name = party = stemmekreds = ''

    # Strategy 1 — __NEXT_DATA__
    if next_data:
        for obj in _search_dict(next_data, 'candidate') + _search_dict(next_data, 'politiker'):
            if isinstance(obj, dict):
                name        = name        or str(obj.get('name','') or obj.get('navn','') or '')
                party       = party       or str(obj.get('party','') or obj.get('parti','') or
                                                  obj.get('partyShortName','') or obj.get('partyLetter','') or '')
                stemmekreds = stemmekreds or str(obj.get('constituency','') or
                                                  obj.get('stemmekreds','') or obj.get('kredsNavn','') or '')
                if name:
                    break

    # Strategy 2 — <title>
    if not name:
        title_tag = soup.find('title')
        if title_tag:
            raw = title_tag.get_text(' ', strip=True)
            m = re.match(r'^(.+?)\s*\((.+?)\)\s*(.*?)\s*[\|–]', raw)
            if m:
                name, party, stemmekreds = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()

    # Strategy 3 — og:title
    if not name:
        og = soup.find('meta', property='og:title')
        if og and og.get('content'):
            m = re.match(r'^(.+?)\s*\((.+?)\)\s*(.*)', og['content'])
            if m:
                name, party, stemmekreds = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()

    # Strategy 4 — first heading
    if not name:
        for tag in ('h1', 'h2'):
            el = soup.find(tag)
            if el:
                name = el.get_text(' ', strip=True)
                break

    return {'candidate_id': candidate_id, 'name': name, 'party': party,
            'stemmekreds': stemmekreds, 'url': url}


# ─────────────────────────────────────────────────────────────────────────────
# Om (About) section
# ─────────────────────────────────────────────────────────────────────────────

KEY_MAP = {
    'uddannelse': 'uddannelse',
    'bopæl': 'bopael', 'bopael': 'bopael', 'bopæl / opvækststed': 'bopael',
    'alder': 'alder', 'år': 'alder', 'fødselsår': 'alder',
    'erhverv': 'erhverv', 'job': 'erhverv', 'stilling': 'erhverv',
    'beskæftigelse': 'erhverv', 'titel': 'erhverv', 'arbejde': 'erhverv',
}


def extract_om_section(soup: BeautifulSoup, next_data: dict = None) -> Dict[str, Any]:
    """
    Extract Om section: uddannelse, bopæl, alder, erhverv, sociale medier.
    Primary: __NEXT_DATA__ JSON.  Fallbacks: <dl> pairs, label/value scan.
    """
    om_data: Dict[str, Any] = {
        'uddannelse': '', 'bopael': '', 'alder': '', 'erhverv': '',
        'sociale_medier': [],
    }

    # Strategy 1 — __NEXT_DATA__
    if next_data:
        for obj in _search_dict(next_data, 'candidate') + _search_dict(next_data, 'politiker'):
            if isinstance(obj, dict):
                om_data['uddannelse'] = om_data['uddannelse'] or str(obj.get('education','') or obj.get('uddannelse','') or '')
                om_data['bopael']     = om_data['bopael']     or str(obj.get('city','') or obj.get('bopael','') or obj.get('bopæl','') or obj.get('residence','') or '')
                om_data['alder']      = om_data['alder']      or str(obj.get('age','') or obj.get('alder','') or obj.get('birthYear','') or '')
                om_data['erhverv']    = om_data['erhverv']    or str(obj.get('occupation','') or obj.get('erhverv','') or obj.get('job','') or obj.get('jobTitle','') or '')
                social = obj.get('socialMedia', []) or obj.get('sociale_medier', []) or []
                if isinstance(social, list):
                    om_data['sociale_medier'].extend([s for s in social if isinstance(s, str) and s not in om_data['sociale_medier']])
                elif isinstance(social, dict):
                    om_data['sociale_medier'].extend([v for v in social.values() if isinstance(v, str) and v not in om_data['sociale_medier']])

    # Strategy 2 — all <dl> on the page
    def _parse_dl(dl):
        for dt, dd in zip(dl.find_all('dt'), dl.find_all('dd')):
            k = dt.get_text(strip=True).lower()
            v = dd.get_text(' ', strip=True)
            for key, canon in KEY_MAP.items():
                if key in k and not om_data[canon]:
                    om_data[canon] = v
                    break

    for dl in soup.find_all('dl'):
        _parse_dl(dl)

    # Strategy 3 — label/value scan (DR wraps items in divs/spans)
    if not any([om_data['uddannelse'], om_data['bopael'], om_data['erhverv']]):
        for el in soup.find_all(['p', 'span', 'div', 'li']):
            text = el.get_text(' ', strip=True)
            for key, canon in KEY_MAP.items():
                if text.lower().startswith(key + ':') or text.lower().startswith(key + ' '):
                    val = text.split(':', 1)[-1].strip() if ':' in text else ''
                    if val and not om_data[canon]:
                        om_data[canon] = val
                    elif not om_data[canon]:
                        nxt = el.find_next_sibling()
                        if nxt:
                            om_data[canon] = nxt.get_text(' ', strip=True)

    # Strategy 4 — social media links anywhere on page
    if not om_data['sociale_medier']:
        for a in soup.find_all('a', href=True):
            h = a['href']
            if any(p in h for p in ['facebook.com', 'twitter.com', 'instagram.com',
                                     'linkedin.com', 'x.com', 'tiktok.com']):
                if h not in om_data['sociale_medier']:
                    om_data['sociale_medier'].append(h)

    return om_data


print('✅ Personal info helpers defined (with __NEXT_DATA__ support)')

✅ Personal info helpers defined (with __NEXT_DATA__ support)


## 6. Extract Candidate Political Answers

DR shows a fixed set of political statements and the candidate's response
(typically **Enig / Delvist enig / Uenig** + a free-text elaboration).  
A *"Vis alle"* button hides answers beyond the first few — we click it before parsing.

In [7]:
# DR Folketing has exactly 25 test questions
N_QUESTIONS = 25

# Vote indicators used by DR stemmeseddel
VOTE_PATTERN = re.compile(
    r'^(enig|delvist enig|uenig|hverken eller|ved ikke)$', re.I
)


# ─────────────────────────────────────────────────────────────────────────────
# Mærkesager
# ─────────────────────────────────────────────────────────────────────────────

def extract_maerkesager(soup: BeautifulSoup, next_data: dict = None) -> List[Dict[str, Any]]:
    """
    Extract mærkesager (policy priority statements).
    Primary: __NEXT_DATA__ JSON.  Fallbacks: DOM heading → ul/li, numbered text.
    Returns [{number, title, description, full_text}, ...]
    """
    priorities: List[Dict[str, Any]] = []

    # Strategy 1 — __NEXT_DATA__
    if next_data:
        for key in ('priorities', 'maerkesager', 'mærkesager', 'prioriteter',
                    'keyIssues', 'issues', 'politics'):
            for hit in _search_dict(next_data, key):
                if isinstance(hit, list) and hit:
                    for idx, item in enumerate(hit, 1):
                        if isinstance(item, dict):
                            title = str(item.get('title','') or item.get('name','') or
                                        item.get('headline','') or item.get('tekst','') or '')
                            desc  = str(item.get('description','') or item.get('text','') or
                                        item.get('body','') or item.get('indhold','') or '')
                            if title:
                                priorities.append({'number': idx, 'title': title,
                                                   'description': desc,
                                                   'full_text': f'{title}: {desc}' if desc else title})
                        elif isinstance(item, str) and item.strip():
                            priorities.append({'number': idx, 'title': item.strip(),
                                               'description': '', 'full_text': item.strip()})
                    if priorities:
                        break
            if priorities:
                break

    # Strategy 2 — heading + adjacent list in DOM
    if not priorities:
        for heading_text in ['mærkesager', 'Mine mærkesager', 'Politiske mærkesager']:
            heading = soup.find(string=re.compile(heading_text, re.I))
            if heading:
                container = heading.find_parent(['section', 'div', 'article', 'nav']) or heading.find_parent()
                if container:
                    ul = container.find_next(['ul', 'ol'])
                    if ul:
                        for idx, li in enumerate(ul.find_all('li'), 1):
                            text = re.sub(r'^\d+\.?\s*', '', li.get_text(' ', strip=True))
                            if not text:
                                continue
                            title = text.split(':')[0].strip() if ':' in text else text.split('.')[0].strip()
                            desc  = text.split(':', 1)[1].strip() if ':' in text else '.'.join(text.split('.')[1:]).strip()
                            priorities.append({'number': idx, 'title': title,
                                               'description': desc, 'full_text': text})
                        if priorities:
                            break

    # Strategy 3 — numbered lines in page text
    if not priorities:
        lines = soup.get_text('\n', strip=True).split('\n')
        current: Optional[Dict] = None
        for line in lines:
            line = line.strip()
            if line.isdigit() and 1 <= int(line) <= 10:
                if current:
                    priorities.append(current)
                current = {'number': int(line), 'title': '', 'description': '', 'full_text': ''}
            elif current and line and not VOTE_PATTERN.match(line):
                current['full_text'] += (' ' if current['full_text'] else '') + line
                if not current['title']:
                    current['title'] = line.split(':')[0].strip() if ':' in line else line
                    current['description'] = line.split(':', 1)[1].strip() if ':' in line else ''
        if current:
            priorities.append(current)

    return priorities


# ─────────────────────────────────────────────────────────────────────────────
# Click 'Vis alle' to expand all 25 test answers
# ─────────────────────────────────────────────────────────────────────────────

def click_vis_alle(driver, max_attempts: int = 3) -> bool:
    """
    Click every 'Vis alle' / 'Vis alle svar' button on the page.
    DR may have one button per section — clicks ALL of them.
    """
    BUTTON_TEXTS = ['vis alle svar', 'vis alle', 'se alle svar', 'se alle', 'vis svar', 'show all']
    clicked = False

    for attempt in range(max_attempts):
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight / 2);')
        time.sleep(0.8)

        # Method A — scan all buttons and click every match
        try:
            buttons = driver.find_elements(By.TAG_NAME, 'button')
            for btn in buttons:
                try:
                    txt = btn.text.strip().lower()
                    if any(t in txt for t in BUTTON_TEXTS):
                        driver.execute_script('arguments[0].scrollIntoView({block:"center"});', btn)
                        time.sleep(0.3)
                        try:
                            btn.click()
                        except ElementClickInterceptedException:
                            driver.execute_script('arguments[0].click();', btn)
                        print(f'  ✓ Clicked: "{btn.text.strip()}"')
                        clicked = True
                        time.sleep(1.5)
                except StaleElementReferenceException:
                    continue
        except Exception:
            pass

        if clicked:
            time.sleep(1)
            return True

        # Method B — XPath text search
        for text in BUTTON_TEXTS:
            try:
                xpath = (
                    f"//button[contains("
                    f"translate(normalize-space(text()),"
                    f"'ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ',"
                    f"'abcdefghijklmnopqrstuvwxyzæøå'),'{text}')]"
                )
                btn = driver.find_element(By.XPATH, xpath)
                driver.execute_script('arguments[0].scrollIntoView({block:"center"});', btn)
                driver.execute_script('arguments[0].click();', btn)
                print(f'  ✓ Clicked via XPath: "{text}"')
                time.sleep(2)
                return True
            except NoSuchElementException:
                continue

        # Method C — aria-expanded=false toggles
        try:
            for toggle in driver.find_elements(By.CSS_SELECTOR, '[aria-expanded="false"]'):
                try:
                    driver.execute_script('arguments[0].click();', toggle)
                    clicked = True
                    time.sleep(0.5)
                except Exception:
                    continue
            if clicked:
                time.sleep(1)
                return True
        except Exception:
            pass

        if attempt < max_attempts - 1:
            time.sleep(1)

    print('  ⚠  No "Vis alle" button found — answers may already be fully visible')
    return False


# ─────────────────────────────────────────────────────────────────────────────
# Extract all 25 test answers
# ─────────────────────────────────────────────────────────────────────────────

def extract_test_answers(driver, next_data: dict = None) -> Dict[int, Dict[str, str]]:
    """
    Extract all N_QUESTIONS (25) political test Q&A pairs.
    Primary: __NEXT_DATA__ JSON.  Fallbacks: data-* attrs, vote-indicator DOM scan.
    Returns {q_num: {question, answer, elaboration}}.
    """
    time.sleep(1.5)
    soup = BeautifulSoup(driver.page_source, 'lxml')
    answers: Dict[int, Dict[str, str]] = {}

    # Strategy 1 — __NEXT_DATA__
    if next_data:
        for key in ('answers', 'testAnswers', 'questions', 'svar', 'testSvar',
                    'candidateAnswers', 'politicalAnswers', 'statements'):
            for hit in _search_dict(next_data, key):
                if isinstance(hit, list) and len(hit) >= 5:
                    for idx, item in enumerate(hit, 1):
                        if isinstance(item, dict):
                            q = str(item.get('question','') or item.get('text','') or
                                    item.get('statement','') or item.get('spørgsmål','') or '')
                            a = str(item.get('answer','') or item.get('choice','') or
                                    item.get('svar','') or item.get('vote','') or '')
                            elab = str(item.get('elaboration','') or item.get('explanation','') or
                                       item.get('begrundelse','') or item.get('uddybning','') or '')
                            if a:
                                answers[idx] = {'question': q, 'answer': a, 'elaboration': elab}
                    if len(answers) >= 5:
                        print(f'  ✓ {len(answers)} answers from __NEXT_DATA__["{key}"]')
                        return answers

    # Strategy 2 — data-* attribute selectors (up to N_QUESTIONS)
    for i in range(1, N_QUESTIONS + 1):
        for sel in [f'[data-question-number="{i}"]', f'[data-question="{i}"]',
                    f'[data-question-id="{i}"]', f'#question-{i}']:
            try:
                el = driver.find_element(By.CSS_SELECTOR, sel)
                q  = el.get_attribute('data-question-text') or ''
                a  = el.text or el.get_attribute('data-answer') or ''
                if a:
                    answers[i] = {'question': q, 'answer': a, 'elaboration': ''}
                    break
            except Exception:
                continue

    # Strategy 3 — vote-indicator DOM scan
    # DR renders each Q&A as a block containing the vote label standalone
    if len(answers) < 5:
        seen: set = set()
        q_num = 1
        VOTE_BROAD = re.compile(r'\b(enig|uenig|delvist)\b', re.I)

        for vote_el in soup.find_all(string=VOTE_PATTERN):
            if q_num > N_QUESTIONS:
                break
            vote_str = vote_el.strip()

            for parent_tag in ['li', 'article', 'section', 'div']:
                parent = vote_el.find_parent(parent_tag)
                if not parent:
                    continue
                full = parent.get_text('\n', strip=True)
                if len(full) < 10 or len(full) > 3000:
                    continue
                key = full[:80]
                if key in seen:
                    break
                seen.add(key)

                lines = [l.strip() for l in full.split('\n') if l.strip()]
                vote_line = vote_str
                q_lines, elab_lines = [], []
                after_vote = False

                for line in lines:
                    if VOTE_PATTERN.match(line):
                        vote_line = line
                        after_vote = True
                    elif not after_vote:
                        q_lines.append(line)
                    else:
                        elab_lines.append(line)

                question = ' '.join(q_lines).strip()
                elab     = ' '.join(elab_lines).strip()

                if question and q_num not in answers:
                    answers[q_num] = {'question': question, 'answer': vote_line, 'elaboration': elab}
                    q_num += 1
                break

    # Strategy 4 — class-name containers with vote words
    if len(answers) < 5:
        cls_pat = re.compile(r'(question|answer|test|statement|svar|kandidat)', re.I)
        q_num   = max(answers.keys(), default=0) + 1
        VOTE_BROAD = re.compile(r'\b(enig|uenig|delvist)\b', re.I)
        for container in soup.find_all(['article', 'section', 'div'], class_=cls_pat):
            if q_num > N_QUESTIONS:
                break
            text   = container.get_text(' ', strip=True)
            vmatch = VOTE_BROAD.search(text)
            if vmatch and 20 < len(text) < 2000 and q_num not in answers:
                answers[q_num] = {
                    'question':    text[:200],
                    'answer':      vmatch.group(0),
                    'elaboration': text[len(vmatch.group(0)):].strip()[:300]
                }
                q_num += 1

    print(f'  ✓ {len(answers)}/{N_QUESTIONS} test answers extracted')
    return answers


print(f'✅ Political-answer helpers defined (N_QUESTIONS={N_QUESTIONS})')

✅ Political-answer helpers defined (N_QUESTIONS=25)


## 7. Main Scraping Function – Folketing v02

In [8]:
def scrape_candidate(candidate_url: str, driver, wait_time: int = 15) -> Dict[str, Any]:
    """
    Full scrape of a single Folketing candidate page.
    Extracts: basic info, Om section, mærkesager,
    and all N_QUESTIONS (25) test answers.
    Primary data source: __NEXT_DATA__ JSON; DOM is fallback.
    """
    print(f'Scraping: {candidate_url}')
    try:
        soup = load_page(driver, candidate_url, wait_time)

        # Extract __NEXT_DATA__ once; pass to all helpers
        next_data = extract_next_data(soup)
        if next_data:
            print('  ✓ __NEXT_DATA__ JSON found')
        else:
            print('  ⚠  __NEXT_DATA__ not found — falling back to DOM')

        # Basic info
        data = extract_basic_info(soup, candidate_url, next_data)
        print(f"  ✓ Basic info: {data['name']} ({data['party']}) — {data['stemmekreds']}")

        # Om section
        om = extract_om_section(soup, next_data)
        data.update(om)
        print(f"  ✓ Om: uddannelse={bool(om['uddannelse'])}, "
              f"bopæl={bool(om['bopael'])}, erhverv={bool(om['erhverv'])}")

        # Mærkesager
        priorities = extract_maerkesager(soup, next_data)
        data['priorities']     = priorities
        data['num_priorities'] = len(priorities)
        print(f'  ✓ Mærkesager: {len(priorities)} found')

        # Click ALL 'Vis alle' buttons, then re-parse __NEXT_DATA__ with updated DOM
        click_vis_alle(driver)
        soup2      = BeautifulSoup(driver.page_source, 'lxml')
        next_data2 = extract_next_data(soup2) or next_data

        # Extract all 25 answers
        answers = extract_test_answers(driver, next_data2)
        data['test_answers']     = answers
        data['num_test_answers'] = len(answers)

        # Flatten all N_QUESTIONS into individual columns
        for i in range(1, N_QUESTIONS + 1):
            qa = answers.get(i, {})
            data[f'svar_{i}_question']    = qa.get('question', '')
            data[f'svar_{i}_answer']      = qa.get('answer', '')
            data[f'svar_{i}_elaboration'] = qa.get('elaboration', '')

        return data

    except RuntimeError as e:
        print(f'  ✗ {e}')
        return {'url': candidate_url, 'error': str(e)}
    except Exception as e:
        import traceback
        print(f'  ✗ Unexpected error: {e}')
        traceback.print_exc()
        return {'url': candidate_url, 'error': str(e)}


print('✅ scrape_candidate() defined')

✅ scrape_candidate() defined


## TEST: Single Candidate — Marcus Vesterager

Run this cell to verify the scraper against the example URL:
`https://www.dr.dk/nyheder/politik/folketingsvalg/din-stemmeseddel/kandidater/527-morten-eg-brautsch`

## DEBUG: Inspect Page Structure
Run this cell to see exactly what the live page contains before running the full scrape.
It prints the `__NEXT_DATA__` top-level keys, sample text from the vote/answer section,
and a snippet of page text — so you can tune selectors if needed.

In [9]:
import pprint

# Re-init driver (run this if the previous session closed it)
try:
    driver.current_url
    print('✓ Existing driver still running')
except Exception:
    driver = setup_driver(headless=True)
    print('✓ New driver started')

# ── Load the example page ─────────────────────────────────────────────────────
debug_soup = load_page(driver, EXAMPLE_CANDIDATE_URL, wait_time=15)

# ── 1. Inspect __NEXT_DATA__ ──────────────────────────────────────────────────
nd = extract_next_data(debug_soup)
if nd:
    print('\n── __NEXT_DATA__ top-level keys:')
    print(list(nd.keys()))
    if 'props' in nd:
        print('── props keys:', list(nd['props'].keys()))
    if 'props' in nd and 'pageProps' in nd['props']:
        pp = nd['props']['pageProps']
        print('── pageProps keys:', list(pp.keys()))
        # Show first 2 levels of each value type
        for k, v in pp.items():
            if isinstance(v, dict):
                print(f'   {k} (dict): {list(v.keys())[:10]}')
            elif isinstance(v, list):
                print(f'   {k} (list[{len(v)}]): {str(v[:1])[:120]}')
            else:
                print(f'   {k}: {str(v)[:120]}')
else:
    print('⚠  No __NEXT_DATA__ found')

# ── 2. Inspect mærkesager ─────────────────────────────────────────────────────
print('\n── Searching __NEXT_DATA__ for mærkesager keys:')
for key in ('priorities','maerkesager','mærkesager','prioriteter','keyIssues','issues'):
    hits = _search_dict(nd, key) if nd else []
    if hits:
        print(f'  Found "{key}": {str(hits[:1])[:200]}')

# ── 3. Inspect answers/questions ─────────────────────────────────────────────
print('\n── Searching __NEXT_DATA__ for answer/question keys:')
for key in ('answers','testAnswers','questions','svar','testSvar','statements','candidateAnswers'):
    hits = _search_dict(nd, key) if nd else []
    if hits:
        for h in hits:
            if isinstance(h, list) and len(h) >= 3:
                print(f'  Found "{key}" (list[{len(h)}]): {str(h[:2])[:300]}')
                break

# ── 4. DOM: vote indicators ───────────────────────────────────────────────────
print('\n── Vote indicator elements in DOM:')
VOTE_BROAD = re.compile(r'\b(enig|uenig|delvist|hverken)\b', re.I)
vote_els = debug_soup.find_all(string=VOTE_PATTERN)
print(f'  Standalone vote labels found: {len(vote_els)}')
if vote_els:
    print(f'  First few: {[e.strip() for e in vote_els[:5]]}')

# ── 5. DOM: page text snippet ─────────────────────────────────────────────────
print('\n── First 3000 chars of page text:')
page_text = debug_soup.get_text('\n', strip=True)
print(page_text[:3000])

✓ Existing driver still running
  ✓ Cookie consent dismissed
⚠  No __NEXT_DATA__ found

── Searching __NEXT_DATA__ for mærkesager keys:

── Searching __NEXT_DATA__ for answer/question keys:

── Vote indicator elements in DOM:
  Standalone vote labels found: 6
  First few: ['Uenig', 'Enig', 'Uenig', 'Enig', 'Uenig']

── First 3000 chars of page text:
Marcus Vesterager (A) Vesterbrokredsen | FV26 | DR
DR passer på dine data
DR indsamler oplysninger om dine besøg ved hjælp af cookies for at måle, hvordan dr.dk bliver brugt, så vi kan udvikle indhold og funktioner. DR indsamler også oplysninger om dine præferencer for at give dig en bedre brugeroplevelse og vise indhold, der er relevant for dig.
DR bruger egne cookies og cookies fra tredjepart. Tredjepart kan anvende cookiedata til målrettet markedsføring på egne og andres platforme. Du kan til- og fravælge cookies herunder og altid se og ændre dine indstillinger i
cookiepolitikken
. Se hvordan DR behandler personoplysninger i
privatlivspo

In [10]:
print('=' * 65)
print('TEST — SINGLE CANDIDATE: Marcus Vesterager')
print('=' * 65)

# Re-init driver if needed
try:
    driver.current_url
except Exception:
    driver = setup_driver(headless=True)
    print('✓ New driver started')

test_data = scrape_candidate(EXAMPLE_CANDIDATE_URL, driver)

print('\n' + '=' * 65)
print('RESULTS')
print('=' * 65)
print(f"Name         : {test_data.get('name', 'N/A')}")
print(f"Party        : {test_data.get('party', 'N/A')}")
print(f"Stemmekreds  : {test_data.get('stemmekreds', 'N/A')}")

print(f"\nOm section:")
print(f"  Uddannelse : {test_data.get('uddannelse', 'N/A')}")
print(f"  Bopæl      : {test_data.get('bopael', 'N/A')}")
print(f"  Alder      : {test_data.get('alder', 'N/A')}")
print(f"  Erhverv    : {test_data.get('erhverv', 'N/A')}")
print(f"  Sociale    : {test_data.get('sociale_medier', [])}")

print(f"\nExtracted:")
print(f"  Mærkesager  : {test_data.get('num_priorities', 0)}")
print(f"  Test answers: {test_data.get('num_test_answers', 0)} / {N_QUESTIONS}")

if test_data.get('priorities'):
    print(f"\nMærkesager:")
    for p in test_data['priorities']:
        print(f"  [{p['number']}] {p.get('title','')[:80]}")
        if p.get('description'):
            print(f"       {p.get('description','')[:100]}")

if test_data.get('test_answers'):
    print(f"\nAll test answers ({len(test_data['test_answers'])} extracted):")
    for q_num in sorted(test_data['test_answers'].keys()):
        qa = test_data['test_answers'][q_num]
        q_short = qa.get('question','')[:70]
        a       = qa.get('answer','')
        elab    = qa.get('elaboration','')[:60]
        elab_str = f' → {elab}' if elab else ''
        print(f"  Q{q_num:02d}: [{a:20s}] {q_short}{elab_str}")

print('=' * 65)

TEST — SINGLE CANDIDATE: Marcus Vesterager
Scraping: https://www.dr.dk/nyheder/politik/folketingsvalg/din-stemmeseddel/kandidater/882-marcus-vesterager


KeyboardInterrupt: 

## 8. Structure and Store Scraped Data

In [ ]:
# Wrap the result in a list so the DataFrame/export code below is reusable
# for both single-candidate and batch scraping.
candidates = [test_data] if 'error' not in test_data else []

# ── Stemmekreds batch scraping (optional) ────────────────────────────────────
def get_stemmekreds_candidate_links(driver, kreds_id: int, wait_time: int = 12) -> List[str]:
    """
    Collect all candidate profile URLs from a stemmekreds listing page.
    URL pattern: /folketingsvalg/din-stemmeseddel/stemmekreds/{kreds_id}
    """
    url = f'{STEMMEKREDS_BASE}/{kreds_id}'
    print(f'\nFetching candidate links from: {url}')
    soup = load_page(driver, url, wait_time)

    # Scroll down to load all candidates
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, 'lxml')

    links = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        # Folketing candidate links: /kandidater/{id}-{slug}  (no /kommune/)
        if '/folketingsvalg/' in href and '/kandidater/' in href and '/kommune/' not in href:
            full = urljoin(BASE_URL, href)
            if full not in links:
                links.append(full)

    print(f'  ✓ Found {len(links)} candidate links')
    return links


def scrape_stemmekreds(driver, kreds_id: int, max_candidates: Optional[int] = None) -> List[Dict[str, Any]]:
    """Scrape all candidates from a stemmekreds."""
    links = get_stemmekreds_candidate_links(driver, kreds_id)
    if max_candidates:
        links = links[:max_candidates]
        print(f'Limiting to {max_candidates} candidates')

    results = []
    for i, link in enumerate(links, 1):
        print(f'\n{"-" * 50}')
        print(f'[{i}/{len(links)}]')
        results.append(scrape_candidate(link, driver))
        time.sleep(2)
    return results


# ── Example batch call (uncomment to use) ────────────────────────────────────
# candidates = scrape_stemmekreds(driver, kreds_id=7, max_candidates=5)
# candidates = scrape_stemmekreds(driver, kreds_id=7)   # full kreds

print(f'\n✅ candidates list ready — {len(candidates)} candidate(s)')

## 9. Export Data to DataFrame

Three outputs:
1. **Main** — one row per candidate with personal info
2. **Mærkesager** — one row per mærkesag
3. **Test answers** — wide format (one row per candidate, one column pair per question)

In [ ]:
ok = [c for c in candidates if 'error' not in c]

# ── 1. Main DataFrame ─────────────────────────────────────────────────────────
df_main = pd.DataFrame([{
    'candidate_id'    : c.get('candidate_id', ''),
    'name'            : c.get('name', ''),
    'party'           : c.get('party', ''),
    'stemmekreds'     : c.get('stemmekreds', ''),
    'uddannelse'      : c.get('uddannelse', ''),
    'bopael'          : c.get('bopael', ''),
    'alder'           : c.get('alder', ''),
    'erhverv'         : c.get('erhverv', ''),
    'sociale_medier'  : ', '.join(c.get('sociale_medier', [])),
    'num_priorities'  : c.get('num_priorities', 0),
    'num_test_answers': c.get('num_test_answers', 0),
    'url'             : c.get('url', ''),
} for c in ok])

print('📊 Main DataFrame:')
display(df_main)

# ── 2. Mærkesager DataFrame ───────────────────────────────────────────────────
mk_rows = []
for c in ok:
    for p in c.get('priorities', []):
        mk_rows.append({
            'candidate_id'        : c.get('candidate_id', ''),
            'name'                : c.get('name', ''),
            'party'               : c.get('party', ''),
            'stemmekreds'         : c.get('stemmekreds', ''),
            'priority_number'     : p.get('number', ''),
            'priority_title'      : p.get('title', ''),
            'priority_description': p.get('description', ''),
            'priority_full_text'  : p.get('full_text', ''),
        })
df_maerkesager = pd.DataFrame(mk_rows)
print('\n📊 Mærkesager DataFrame:')
display(df_maerkesager)

# ── 3. Test answers — wide format (N_QUESTIONS × 3 cols each) ─────────────────
def _c_answers(c):
    row = {
        'candidate_id': c.get('candidate_id', ''),
        'name'        : c.get('name', ''),
        'party'       : c.get('party', ''),
        'stemmekreds' : c.get('stemmekreds', ''),
    }
    for i in range(1, N_QUESTIONS + 1):
        row[f'svar_{i}_question']    = c.get(f'svar_{i}_question', '')
        row[f'svar_{i}_answer']      = c.get(f'svar_{i}_answer', '')
        row[f'svar_{i}_elaboration'] = c.get(f'svar_{i}_elaboration', '')
    return row

df_svars = pd.DataFrame([_c_answers(c) for c in ok])
print(f'\n📊 Test Answers DataFrame ({N_QUESTIONS} questions × 3 cols each):')
# Show answer column only for a compact preview
answer_preview = ['name', 'party'] + [f'svar_{i}_answer' for i in range(1, N_QUESTIONS + 1)]
existing = [col for col in answer_preview if col in df_svars.columns]
display(df_svars[existing])

## 10. Save Files

In [ ]:
import os
OUT_DIR = os.path.dirname(os.path.abspath('__file__'))  # same dir as notebook

df_main.to_csv(os.path.join(OUT_DIR, 'candidates_main.csv'), index=False, encoding='utf-8')
df_svars.to_csv(os.path.join(OUT_DIR, 'candidates_svars.csv'), index=False, encoding='utf-8')

if not df_maerkesager.empty:
    df_maerkesager.to_csv(os.path.join(OUT_DIR, 'candidates_maerkesager.csv'), index=False, encoding='utf-8')

with open(os.path.join(OUT_DIR, 'candidates_raw.json'), 'w', encoding='utf-8') as f:
    json.dump(candidates, f, ensure_ascii=False, indent=2)

print('\n✅ Saved:')
print('  candidates_main.csv')
print('  candidates_svars.csv')
if not df_maerkesager.empty:
    print('  candidates_maerkesager.csv')
print('  candidates_raw.json')

## 11. Close WebDriver

In [ ]:
driver.quit()
print('✅ WebDriver closed')
print('\n' + '=' * 65)
print('SCRAPING COMPLETE')
print('=' * 65)